# SQLCoder 7B — Fine-Tuning on Colab TPU v5e-1

Run each cell top to bottom. Do not skip any cell.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

packages = [
    'transformers==4.40.0',
    'peft==0.10.0',
    'accelerate==0.29.0',
    'datasets==2.18.0',
    'trl==0.8.6',
    'loguru',
]
for p in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])

print('✅ All packages installed')

In [ ]:
# Cell 2 — Confirm TPU
import torch
import torch_xla
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f'PyTorch  : {torch.__version__}')
print(f'XLA      : {torch_xla.__version__}')
print(f'Device   : {device}')
print('✅ TPU confirmed')

In [ ]:
# Cell 3 — Upload your dataset.csv
from google.colab import files
import os, json
from pathlib import Path

print('Select your dataset.csv file...')
uploaded = files.upload()   # click Choose Files → pick dataset.csv

fname = list(uploaded.keys())[0]
print(f'Uploaded: {fname}')

# Rename to standard name if needed
if fname != 'dataset.csv':
    os.rename(fname, 'dataset.csv')
    print('Renamed to dataset.csv')

print('✅ File ready')

In [ ]:
# Cell 4 — Load, clean, split, write JSONL
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('dataset.csv')
print(f'Rows: {len(df):,}  Columns: {list(df.columns)}')

# Clean
required = ['structured_input', 'sql', 'sql_type']
df = df.dropna(subset=required)
df['structured_input'] = df['structured_input'].astype(str).str.strip()
df['sql']              = df['sql'].astype(str).str.strip()
df['sql_type']         = df['sql_type'].astype(str).str.strip().str.upper()
df = df[(df['structured_input'].str.len() > 0) & (df['sql'].str.len() > 0)].reset_index(drop=True)
print(f'After clean: {len(df):,}')

# Merge rare types
rare = df['sql_type'].value_counts()
rare = rare[rare < 3].index.tolist()
if rare:
    df.loc[df['sql_type'].isin(rare), 'sql_type'] = 'OTHER'
    print(f'Merged rare types → OTHER: {rare}')

# Split
def safe_split(data, test_size, seed, label=''):
    try:
        return train_test_split(data, test_size=test_size, stratify=data['sql_type'], random_state=seed)
    except ValueError:
        print(f'  ⚠ {label}: random split')
        return train_test_split(data, test_size=test_size, random_state=seed)

train_df, temp_df = safe_split(df, 0.20, 42, 'train/temp')
val_df,   test_df = safe_split(temp_df, 0.50, 42, 'val/test')
print(f'Train:{len(train_df):,}  Val:{len(val_df):,}  Test:{len(test_df):,}')

# Write JSONL
TRAIN_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n{sql}'
INFER_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n'

def to_record(row, infer=False):
    tmpl = INFER_TMPL if infer else TRAIN_TMPL
    return {'text': tmpl.format(**row), 'sql': row['sql'], 'sql_type': row['sql_type'], 'structured_input': row['structured_input']}

def write_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        for r in records: f.write(json.dumps(r) + '\n')
    print(f'  Wrote {len(records):,} → {path}')

write_jsonl([to_record(r)           for r in train_df.to_dict('records')], 'data/train.jsonl')
write_jsonl([to_record(r)           for r in val_df.to_dict('records')],   'data/val.jsonl')
write_jsonl([to_record(r)           for r in test_df.to_dict('records')],  'data/test.jsonl')
write_jsonl([to_record(r, infer=True) for r in test_df.to_dict('records')],'data/test_inference.jsonl')
print('✅ Data ready')

In [ ]:
# Cell 5 — Load tokenizer
from transformers import AutoTokenizer

MODEL_NAME  = 'defog/sqlcoder-7b-2'
MAX_SEQ_LEN = 512

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'✅ Tokenizer ready — vocab: {tokenizer.vocab_size:,}')

In [ ]:
# Cell 6 — Load model on TPU
import torch
import torch_xla.core.xla_model as xm
from transformers import AutoModelForCausalLM

device = xm.xla_device()

print(f'Loading SQLCoder 7B → TPU (bfloat16) ...')
print('Downloading ~14GB on first run ...')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # bfloat16 is native on TPU v5
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = model.to(device)
model.config.use_cache      = False
model.config.pretraining_tp = 1

total = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total/1e9:.2f}B')
print(f'Dtype      : {next(model.parameters()).dtype}')
print('✅ Model on TPU')

In [ ]:
# Cell 7 — Apply LoRA
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,                          # higher rank than Mac — TPU can handle it
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # more targets on TPU
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied')

In [ ]:
# Cell 8 — Tokenise dataset
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return Dataset.from_list([json.loads(l) for l in f])

train_ds = load_jsonl('data/train.jsonl')
val_ds   = load_jsonl('data/val.jsonl')

def tokenise(examples):
    out = tokenizer(examples['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    out['labels'] = out['input_ids'].copy()
    return out

remove_cols = [c for c in train_ds.column_names if c != 'text']
train_tok = train_ds.map(tokenise, batched=True, remove_columns=remove_cols, desc='Train')
val_tok   = val_ds.map(tokenise,   batched=True, remove_columns=remove_cols, desc='Val')

lens = [len(x) for x in train_tok['input_ids']]
print(f'Token lengths — min:{min(lens)} max:{max(lens)} avg:{int(np.mean(lens))}')
print(f'Train:{len(train_tok):,}  Val:{len(val_tok):,}')
print('✅ Tokenisation done')

In [ ]:
import csv, time
from pathlib import Path
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)

Path('logs').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)

log_file   = open('logs/train_log.csv', 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['step', 'train_loss', 'eval_loss', 'lr', 'epoch'])

class CsvLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_writer.writerow([
                state.global_step,
                logs.get('loss'),
                logs.get('eval_loss'),
                logs.get('learning_rate'),
                logs.get('epoch'),
            ])
            log_file.flush()

training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy='steps',   # correct for 4.40
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=True,
    fp16=False,
    dataloader_pin_memory=False,
    report_to='none',
    tpu_num_cores=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,           # correct for 4.40
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[CsvLogger()],
)

print('🚀 Training started on TPU v5e-1')
start = time.time()
try:
    trainer.train()
finally:
    elapsed = int(time.time() - start)
    log_file.close()
    h, m = divmod(elapsed // 60, 60)
    print(f'\n✅ Done in {h}h {m}m')

In [ ]:
# Cell 10 — Save adapter
import pandas as pd

model.save_pretrained('checkpoints/best_adapter')
tokenizer.save_pretrained('checkpoints/best_adapter')
print('✅ Adapter saved → checkpoints/best_adapter')

log_df = pd.read_csv('logs/train_log.csv').dropna(subset=['train_loss'])
print(f'Final train loss : {log_df["train_loss"].iloc[-1]:.4f}')
print(f'Best eval loss   : {log_df["eval_loss"].dropna().min():.4f}')
print()
print('✅ Paste output in chat — I will give you the eval notebook')

In [ ]:
# Cell 11 — Save checkpoint to Google Drive (so it survives session disconnect)
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/sqlcoder_checkpoints'
shutil.copytree('checkpoints/best_adapter', dest, dirs_exist_ok=True)
print(f'✅ Checkpoint saved to Google Drive → {dest}')
print('Your work is safe even if Colab disconnects')